In [1]:
import pandas as pd

In [2]:
# Notebook 위치를 기준으로 상대경로를 사용해 원본 데이터를 불러옵니다.
df = pd.read_csv("../data/raw/Car details v3.csv")

In [3]:
# 전처리 전 데이터의 행과 열 개수를 확인합니다.
df.shape

(8128, 13)

In [4]:
# 전처리 전 완전히 동일한 중복 행의 개수를 확인합니다.
df.duplicated().sum()

np.int64(1202)

In [5]:
# 원본 df는 유지하고, 각 중복 그룹의 첫 번째 행만 남긴 별도 DataFrame을 만듭니다.
df_clean = df.drop_duplicates().copy()

In [6]:
# 중복 제거 후 데이터의 행과 열 개수를 확인합니다.
df_clean.shape

(6926, 13)

In [7]:
# 중복 제거로 실제 삭제된 행의 개수를 계산합니다.
len(df) - len(df_clean)

1202

In [8]:
# 중복 제거 후 완전히 동일한 중복 행이 남아 있는지 확인합니다.
df_clean.duplicated().sum()

np.int64(0)

In [9]:
# 중복 제거 후 각 컬럼의 결측치 개수를 확인합니다.
df_clean.isna().sum()

name               0
year               0
selling_price      0
km_driven          0
fuel               0
seller_type        0
transmission       0
owner              0
mileage          208
engine           208
max_power        205
torque           209
seats            208
dtype: int64

In [10]:
# 결측치가 있는 5개 컬럼을 지정합니다.
missing_columns = [
    "mileage",
    "engine",
    "max_power",
    "torque",
    "seats",
]

# 각 행에서 결측인 컬럼 수와 그 분포를 확인합니다.
missing_count_per_row_clean = df_clean[missing_columns].isna().sum(axis=1)

missing_count_per_row_clean.value_counts().sort_index()

0    6717
1       1
4       3
5     205
Name: count, dtype: int64

In [11]:
# 5개 컬럼이 모두 동시에 결측인 행 수를 확인합니다.
df_clean[missing_columns].isna().all(axis=1).sum()

np.int64(205)

In [12]:
# 5개 중 정확히 4개 컬럼이 결측인 행을 모두 확인합니다.
df_clean.loc[missing_count_per_row_clean == 4]

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,torque,seats
575,Maruti Alto K10 LXI,2011,204999,97500,Petrol,Individual,Manual,First Owner,NaN,NaN,0,NaN,NaN
1442,Maruti Swift Dzire VDI Optional,2017,589000,41232,Diesel,Dealer,Manual,First Owner,NaN,NaN,0,NaN,NaN
2549,Tata Indica Vista Quadrajet LS,2012,240000,70000,Diesel,Individual,Manual,First Owner,NaN,NaN,0,NaN,NaN


In [13]:
# 5개 중 정확히 1개 컬럼만 결측인 행을 모두 확인합니다.
df_clean.loc[missing_count_per_row_clean == 1]

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,torque,seats
4933,Maruti Omni CNG,2000,80000,100000,CNG,Individual,Manual,Second Owner,10.9 km/kg,796 CC,bhp,NaN,8.0


In [14]:
# 5개 컬럼 중 하나라도 결측인 전체 행 수를 확인합니다.
df_clean[missing_columns].isna().any(axis=1).sum()

np.int64(209)

In [15]:
# mileage의 숫자 부분과 단위 표기를 확인합니다.
mileage_text = df_clean["mileage"].dropna().astype(str).str.strip()

mileage_parts = mileage_text.str.extract(
    r"^(?P<number>[0-9]+(?:\.[0-9]+)?)\s*(?P<unit>.*)$"
)

mileage_parts["unit"].value_counts(dropna=False)

unit
kmpl     6631
km/kg      87
Name: count, dtype: int64

In [16]:
# mileage의 숫자 부분 추출 실패 행 수를 확인합니다.
mileage_failure_mask = mileage_parts["number"].isna()
mileage_failure_count = mileage_failure_mask.sum()

mileage_failure_count

np.int64(0)

In [17]:
# mileage의 숫자 부분 추출 실패 행을 확인합니다.
mileage_failure_index = mileage_parts.index[mileage_failure_mask]
df_clean.loc[mileage_failure_index, ["name", "mileage"]]

,name,mileage


In [18]:
# engine의 숫자 부분과 단위 표기를 확인합니다.
engine_text = df_clean["engine"].dropna().astype(str).str.strip()

engine_parts = engine_text.str.extract(
    r"^(?P<number>[0-9]+(?:\.[0-9]+)?)\s*(?P<unit>.*)$"
)

engine_parts["unit"].value_counts(dropna=False)

unit
CC    6718
Name: count, dtype: int64

In [19]:
# engine의 숫자 부분 추출 실패 행 수를 확인합니다.
engine_failure_mask = engine_parts["number"].isna()
engine_failure_count = engine_failure_mask.sum()

engine_failure_count

np.int64(0)

In [20]:
# engine의 숫자 부분 추출 실패 행을 확인합니다.
engine_failure_index = engine_parts.index[engine_failure_mask]
df_clean.loc[engine_failure_index, ["name", "engine"]]

,name,engine


In [21]:
# max_power의 숫자 부분과 단위 표기를 확인합니다.
max_power_text = df_clean["max_power"].dropna().astype(str).str.strip()

max_power_parts = max_power_text.str.extract(
    r"^(?P<number>[0-9]+(?:\.[0-9]+)?)\s*(?P<unit>.*)$"
)

max_power_parts["unit"].value_counts(dropna=False)

unit
bhp    6717
          3
NaN       1
Name: count, dtype: int64

In [22]:
# max_power의 숫자 부분 추출 실패 행 수를 확인합니다.
max_power_failure_mask = max_power_parts["number"].isna()
max_power_failure_count = max_power_failure_mask.sum()

max_power_failure_count

np.int64(1)

In [23]:
# max_power의 숫자 부분 추출 실패 행을 모두 확인합니다.
max_power_failure_index = max_power_parts.index[max_power_failure_mask]
df_clean.loc[max_power_failure_index, ["name", "max_power"]]

,name,max_power
4933,Maruti Omni CNG,bhp


In [24]:
# 숫자는 추출되지만 max_power 단위가 빈 문자열인 행 수를 확인합니다.
max_power_empty_unit_mask = (
    max_power_parts["number"].notna()
    & max_power_parts["unit"].eq("")
)
max_power_empty_unit_count = max_power_empty_unit_mask.sum()

max_power_empty_unit_count

np.int64(3)

In [25]:
# 숫자는 추출되지만 max_power 단위가 빈 문자열인 행을 모두 확인합니다.
max_power_empty_unit_index = max_power_parts.index[max_power_empty_unit_mask]
df_clean.loc[max_power_empty_unit_index, ["name", "year", "max_power"]]

,name,year,max_power
575,Maruti Alto K10 LXI,2011,0
1442,Maruti Swift Dzire VDI Optional,2017,0
2549,Tata Indica Vista Quadrajet LS,2012,0


In [26]:
# max_power에서 추출한 숫자 중 0 이하인 행을 확인합니다.
max_power_number = pd.to_numeric(max_power_parts["number"], errors="coerce")
max_power_non_positive_mask = max_power_number.le(0)
max_power_non_positive_index = max_power_number.index[max_power_non_positive_mask]

df_clean.loc[max_power_non_positive_index, ["name", "year", "max_power"]]

,name,year,max_power
575,Maruti Alto K10 LXI,2011,0
1442,Maruti Swift Dzire VDI Optional,2017,0
2549,Tata Indica Vista Quadrajet LS,2012,0


In [27]:
# torque에서 사용되는 단위 유형을 확인합니다.
torque_text = df_clean["torque"].dropna().astype(str).str.strip()
torque_lower = torque_text.str.lower()

torque_nm_mask = torque_lower.str.contains("nm", regex=False)
torque_kgm_mask = torque_lower.str.contains("kgm", regex=False)
torque_other_mask = ~torque_nm_mask & ~torque_kgm_mask

pd.Series({
    "전체 non-null": len(torque_text),
    "nm 포함": torque_nm_mask.sum(),
    "kgm 포함": torque_kgm_mask.sum(),
    "nm과 kgm 미포함": torque_other_mask.sum(),
})

전체 non-null    6717
nm 포함          6227
kgm 포함          481
nm과 kgm 미포함      10
dtype: int64

In [28]:
# nm과 kgm 어느 것도 포함하지 않은 torque 고유값과 등장 횟수를 확인합니다.
torque_text[torque_other_mask].value_counts()

torque
210 / 1900           7
250@ 1250-5000rpm    1
510@ 1600-2400       1
110(11.2)@ 4800      1
Name: count, dtype: int64

In [29]:
# torque의 주요 표기 방식별 행 수를 확인합니다.
torque_at_sign_mask = torque_text.str.contains("@", regex=False)
torque_at_word_mask = torque_lower.str.contains("at", regex=False)
torque_range_mask = torque_text.str.contains("-", regex=False)

pd.Series({
    "@ 포함": torque_at_sign_mask.sum(),
    "at 포함": torque_at_word_mask.sum(),
    "- 포함": torque_range_mask.sum(),
})

@ 포함     6493
at 포함     212
- 포함     2196
dtype: int64

In [30]:
# @ 표기가 있는 torque 고유값 예시를 최대 10개 확인합니다.
torque_text[torque_at_sign_mask].drop_duplicates().head(10)

0            190Nm@ 2000rpm
1       250Nm@ 1500-2500rpm
2     12.7@ 2,700(kgm@ rpm)
4     11.5@ 4,500(kgm@ rpm)
5         113.75nm@ 4000rpm
6      7.8@ 4,500(kgm@ rpm)
7             59Nm@ 2500rpm
8       170Nm@ 1800-2400rpm
9            160Nm@ 2000rpm
10           248Nm@ 2250rpm
Name: torque, dtype: str

In [31]:
# at 표기가 있는 torque 고유값 예시를 최대 10개 확인합니다.
torque_text[torque_at_word_mask].drop_duplicates().head(10)

3      22.4 kgm at 1750-2750rpm
109           96 Nm at 3000 rpm
149          250 Nm at 2750 rpm
190           146Nm at 4800 rpm
193        14.9 KGM at 3000 RPM
226       11.4 kgm at 4,000 rpm
286      180 Nm at 1440-1500rpm
472         135 Nm at 2500  rpm
474     24 KGM at 1900-2750 RPM
641     260 Nm at 1800-2200 rpm
Name: torque, dtype: str

In [32]:
# 회전수 범위처럼 보이는 - 표기가 있는 torque 고유값 예시를 최대 10개 확인합니다.
torque_text[torque_range_mask].drop_duplicates().head(10)

1          250Nm@ 1500-2500rpm
3     22.4 kgm at 1750-2750rpm
8          170Nm@ 1800-2400rpm
15         115Nm@ 3500-3600rpm
19       219.7Nm@ 1500-2750rpm
39         320Nm@ 1700-2700rpm
41         250Nm@ 1750-2500rpm
47         343Nm@ 1400-3400rpm
48         200Nm@ 1400-3400rpm
49         200Nm@ 1250-4000rpm
Name: torque, dtype: str

In [33]:
# 조사 후에도 df_clean의 크기가 그대로인지 확인합니다.
df_clean.shape

(6926, 13)

In [34]:
# 조사 후에도 완전 중복 행이 없는지 확인합니다.
df_clean.duplicated().sum()

np.int64(0)

In [35]:
# nm과 kgm을 동시에 포함하는 torque 행 수를 확인합니다.
both_unit_mask = (
    torque_lower.str.contains("nm", na=False)
    & torque_lower.str.contains("kgm", na=False)
)

both_unit_mask.sum()

np.int64(1)

In [36]:
# nm과 kgm을 동시에 포함하는 원본 행을 확인합니다.
both_unit_index = torque_lower.index[both_unit_mask]
df_clean.loc[both_unit_index, ["name", "year", "torque"]]

,name,year,torque
778,Ford Endeavour Hurricane Limited Edition,2013,380Nm(38.7kgm)@ 2500rpm


In [37]:
# torque 단위 조건이 서로 겹치지 않도록 네 그룹의 행 수를 확인합니다.
nm_only_mask = (
    torque_lower.str.contains("nm", na=False)
    & ~torque_lower.str.contains("kgm", na=False)
)
kgm_only_mask = (
    ~torque_lower.str.contains("nm", na=False)
    & torque_lower.str.contains("kgm", na=False)
)
neither_unit_mask = (
    ~torque_lower.str.contains("nm", na=False)
    & ~torque_lower.str.contains("kgm", na=False)
)

exclusive_unit_counts = pd.Series({
    "nm만 포함": nm_only_mask.sum(),
    "kgm만 포함": kgm_only_mask.sum(),
    "nm과 kgm 모두 포함": both_unit_mask.sum(),
    "nm과 kgm 모두 미포함": neither_unit_mask.sum(),
})
exclusive_unit_counts.loc["네 그룹 합계"] = exclusive_unit_counts.sum()

exclusive_unit_counts

nm만 포함            6226
kgm만 포함            480
nm과 kgm 모두 포함        1
nm과 kgm 모두 미포함      10
네 그룹 합계           6717
dtype: int64

In [38]:
# 확인 후에도 df_clean의 크기가 그대로인지 확인합니다.
df_clean.shape

(6926, 13)

In [39]:
# 확인 후에도 완전 중복 행이 없는지 확인합니다.
df_clean.duplicated().sum()

np.int64(0)

In [40]:
# 기존 at 조건과 정확한 공백 포함 at 조건의 행 수를 비교합니다.
torque_exact_at_mask = torque_lower.str.contains(" at ", regex=False)

pd.Series({
    "기존 \"at\" 포함": torque_at_word_mask.sum(),
    "정확한 \" at \" 포함": torque_exact_at_mask.sum(),
})

기존 "at" 포함       212
정확한 " at " 포함    212
dtype: int64

In [41]:
# 두 at 조건의 결과가 다른 행 수를 확인합니다.
torque_at_difference_mask = torque_at_word_mask != torque_exact_at_mask

torque_at_difference_mask.sum()

np.int64(0)

In [42]:
# 두 at 조건에서 차이가 발생하는 torque 고유값과 등장 횟수를 확인합니다.
torque_text[torque_at_difference_mask].value_counts()

Series([], Name: count, dtype: int64)

In [43]:
# 확인 후에도 df_clean의 크기가 그대로인지 확인합니다.
df_clean.shape

(6926, 13)

In [44]:
# 확인 후에도 완전 중복 행이 없는지 확인합니다.
df_clean.duplicated().sum()

np.int64(0)

In [45]:
# mileage 숫자 부분을 확인용 숫자 Series로 변환합니다.
mileage_number = pd.to_numeric(
    mileage_parts["number"],
    errors="coerce",
)

mileage_unit = mileage_parts["unit"]

In [46]:
# 연료 유형과 mileage 단위의 관계를 확인합니다.
mileage_unit_check = pd.DataFrame({
    "fuel": df_clean.loc[mileage_unit.index, "fuel"],
    "mileage_unit": mileage_unit,
})

pd.crosstab(
    mileage_unit_check["fuel"],
    mileage_unit_check["mileage_unit"],
)

mileage_unit,km/kg,kmpl
fuel,,
CNG,52,0
Diesel,0,3658
LPG,35,0
Petrol,0,2973


In [47]:
# 각 연료 유형에서 나타나는 mileage 단위 종류 수를 확인합니다.
mileage_unit_check.groupby("fuel")["mileage_unit"].nunique()

fuel
CNG       1
Diesel    1
LPG       1
Petrol    1
Name: mileage_unit, dtype: int64

In [48]:
# 전체 non-null mileage 숫자값의 기본 분포를 확인합니다.
mileage_number.describe()

count    6718.00000
mean       19.46531
std         4.04915
min         0.00000
25%        16.80000
50%        19.44000
75%        22.50000
max        42.00000
Name: number, dtype: float64

In [49]:
# mileage 숫자값과 단위를 묶어 단위별 분포를 확인합니다.
mileage_numeric_check = pd.DataFrame({
    "mileage_number": mileage_number,
    "mileage_unit": mileage_unit,
})

mileage_numeric_check.groupby("mileage_unit")["mileage_number"].agg(
    ["count", "min", "median", "mean", "max"]
)

,count,min,median,mean,max
mileage_unit,,,,,
km/kg,87,10.9,21.94,21.810805,33.44
kmpl,6631,0.0,19.40,19.434536,42.00


In [50]:
# mileage 숫자값이 0 이하인 행 수를 확인합니다.
mileage_non_positive_mask = mileage_number.le(0)
mileage_non_positive_count = mileage_non_positive_mask.sum()

mileage_non_positive_count

np.int64(15)

In [51]:
# mileage 숫자값이 0 이하인 원본 행을 확인합니다.
mileage_non_positive_index = mileage_number.index[mileage_non_positive_mask]
df_clean.loc[mileage_non_positive_index, ["name", "year", "fuel", "mileage"]]

,name,year,fuel,mileage
644,Tata Indica Vista Aura Safire Anniversary Edition,2009,Petrol,0.0 kmpl
785,Hyundai Santro Xing GL,2009,Petrol,0.0 kmpl
1649,Hyundai Santro Xing GL,2008,Petrol,0.0 kmpl
1676,Mercedes-Benz M-Class ML 350 4Matic,2011,Diesel,0.0 kmpl
2137,Land Rover Freelander 2 TD4 HSE,2013,Diesel,0.0 kmpl
2366,Hyundai Santro Xing (Non-AC),2010,Petrol,0.0 kmpl
2725,Hyundai Santro Xing (Non-AC),2013,Petrol,0.0 kmpl
5276,Hyundai Santro Xing GL,2008,Petrol,0.0 kmpl
5843,Volkswagen Polo GT TSI BSIV,2014,Petrol,0.0 kmpl
5846,Volkswagen Polo GT TSI BSIV,2014,Petrol,0.0 kmpl


In [52]:
# non-null mileage 문자열 중 숫자 변환 실패 수를 확인합니다.
mileage_number.isna().sum()

np.int64(0)

In [53]:
# 조사 후에도 df_clean의 크기가 그대로인지 확인합니다.
df_clean.shape

(6926, 13)

In [54]:
# 조사 후에도 완전 중복 행이 없는지 확인합니다.
df_clean.duplicated().sum()

np.int64(0)

In [55]:
# mileage가 0 이하인 원본 행을 다시 지정합니다.
zero_mileage_index = mileage_number.index[mileage_number.le(0)]

zero_mileage_rows = df_clean.loc[
    zero_mileage_index,
    ["name", "year", "fuel", "mileage"],
]

zero_mileage_rows

,name,year,fuel,mileage
644,Tata Indica Vista Aura Safire Anniversary Edition,2009,Petrol,0.0 kmpl
785,Hyundai Santro Xing GL,2009,Petrol,0.0 kmpl
1649,Hyundai Santro Xing GL,2008,Petrol,0.0 kmpl
1676,Mercedes-Benz M-Class ML 350 4Matic,2011,Diesel,0.0 kmpl
2137,Land Rover Freelander 2 TD4 HSE,2013,Diesel,0.0 kmpl
2366,Hyundai Santro Xing (Non-AC),2010,Petrol,0.0 kmpl
2725,Hyundai Santro Xing (Non-AC),2013,Petrol,0.0 kmpl
5276,Hyundai Santro Xing GL,2008,Petrol,0.0 kmpl
5843,Volkswagen Polo GT TSI BSIV,2014,Petrol,0.0 kmpl
5846,Volkswagen Polo GT TSI BSIV,2014,Petrol,0.0 kmpl


In [56]:
# 0 mileage가 차종별로 몇 건인지 확인합니다.
zero_mileage_rows["name"].value_counts()

name
Hyundai Santro Xing GL                               5
Hyundai Santro Xing (Non-AC)                         2
Volkswagen Polo GT TSI BSIV                          2
Tata Indica Vista Aura Safire Anniversary Edition    1
Mercedes-Benz M-Class ML 350 4Matic                  1
Land Rover Freelander 2 TD4 HSE                      1
Mahindra Bolero Pik-Up FB 1.7T                       1
Mahindra Bolero Pik-Up CBC 1.7T                      1
Mercedes-Benz GLC 220d 4MATIC                        1
Name: count, dtype: int64

In [57]:
# mileage 숫자값이 0보다 큰 행만 비교용으로 준비합니다.
positive_mileage_index = mileage_number.index[mileage_number.gt(0)]

positive_mileage_rows = df_clean.loc[
    positive_mileage_index,
    ["name", "year", "fuel", "mileage"],
]

In [58]:
# 각 0 mileage 행과 같은 name의 양수 mileage를 확인합니다.
for idx, row in zero_mileage_rows.iterrows():
    same_name = positive_mileage_rows[
        positive_mileage_rows["name"].eq(row["name"])
    ]

    print("index: {}".format(idx))
    print("name: {}".format(row["name"]))
    print("year: {}".format(row["year"]))
    print("fuel: {}".format(row["fuel"]))
    print("현재 mileage: {}".format(row["mileage"]))
    print("같은 name의 양수 mileage:")
    print(same_name["mileage"].value_counts())
    print("-" * 50)

index: 644
name: Tata Indica Vista Aura Safire Anniversary Edition
year: 2009
fuel: Petrol
현재 mileage: 0.0 kmpl
같은 name의 양수 mileage:
Series([], Name: count, dtype: int64)
--------------------------------------------------
index: 785
name: Hyundai Santro Xing GL
year: 2009
fuel: Petrol
현재 mileage: 0.0 kmpl
같은 name의 양수 mileage:
Series([], Name: count, dtype: int64)
--------------------------------------------------
index: 1649
name: Hyundai Santro Xing GL
year: 2008
fuel: Petrol
현재 mileage: 0.0 kmpl
같은 name의 양수 mileage:
Series([], Name: count, dtype: int64)
--------------------------------------------------
index: 1676
name: Mercedes-Benz M-Class ML 350 4Matic
year: 2011
fuel: Diesel
현재 mileage: 0.0 kmpl
같은 name의 양수 mileage:
Series([], Name: count, dtype: int64)
--------------------------------------------------
index: 2137
name: Land Rover Freelander 2 TD4 HSE
year: 2013
fuel: Diesel
현재 mileage: 0.0 kmpl
같은 name의 양수 mileage:
Series([], Name: count, dtype: int64)
------------------------

In [59]:
# 각 0 mileage 행과 같은 name과 year의 양수 mileage를 확인합니다.
for idx, row in zero_mileage_rows.iterrows():
    same_name_year = positive_mileage_rows[
        positive_mileage_rows["name"].eq(row["name"])
        & positive_mileage_rows["year"].eq(row["year"])
    ]

    print("index: {}".format(idx))
    print("name: {}".format(row["name"]))
    print("year: {}".format(row["year"]))
    print("같은 name + year의 양수 mileage 기록 존재: {}".format(
        not same_name_year.empty
    ))
    print("같은 name + year의 양수 mileage:")
    print(same_name_year["mileage"].value_counts())
    print("-" * 50)

index: 644
name: Tata Indica Vista Aura Safire Anniversary Edition
year: 2009
같은 name + year의 양수 mileage 기록 존재: False
같은 name + year의 양수 mileage:
Series([], Name: count, dtype: int64)
--------------------------------------------------
index: 785
name: Hyundai Santro Xing GL
year: 2009
같은 name + year의 양수 mileage 기록 존재: False
같은 name + year의 양수 mileage:
Series([], Name: count, dtype: int64)
--------------------------------------------------
index: 1649
name: Hyundai Santro Xing GL
year: 2008
같은 name + year의 양수 mileage 기록 존재: False
같은 name + year의 양수 mileage:
Series([], Name: count, dtype: int64)
--------------------------------------------------
index: 1676
name: Mercedes-Benz M-Class ML 350 4Matic
year: 2011
같은 name + year의 양수 mileage 기록 존재: False
같은 name + year의 양수 mileage:
Series([], Name: count, dtype: int64)
--------------------------------------------------
index: 2137
name: Land Rover Freelander 2 TD4 HSE
year: 2013
같은 name + year의 양수 mileage 기록 존재: False
같은 name + year의 양수 mileag

In [60]:
# 0 mileage 행 중 비교 가능한 행 수를 집계합니다.
positive_names = set(positive_mileage_rows["name"])
positive_name_year_pairs = set(zip(
    positive_mileage_rows["name"],
    positive_mileage_rows["year"],
))

same_name_available = zero_mileage_rows["name"].isin(positive_names)
same_name_year_available = pd.Series(
    [
        (row["name"], row["year"]) in positive_name_year_pairs
        for _, row in zero_mileage_rows.iterrows()
    ],
    index=zero_mileage_rows.index,
)

pd.Series({
    "같은 name으로 비교 가능": same_name_available.sum(),
    "같은 name + year로 비교 가능": same_name_year_available.sum(),
    "같은 name에서도 비교 값 없음": (~same_name_available).sum(),
})

같은 name으로 비교 가능           0
같은 name + year로 비교 가능     0
같은 name에서도 비교 값 없음       15
dtype: int64

In [61]:
# 확인 후에도 df_clean의 크기가 그대로인지 확인합니다.
df_clean.shape

(6926, 13)

In [62]:
# 확인 후에도 완전 중복 행이 없는지 확인합니다.
df_clean.duplicated().sum()

np.int64(0)

In [63]:
# 실제 전처리를 적용할 별도 DataFrame을 만듭니다.
df_preprocessed = df_clean.copy()

In [64]:
# mileage에서 숫자 부분만 추출해 숫자형으로 변환합니다.
df_preprocessed["mileage"] = pd.to_numeric(
    df_preprocessed["mileage"].str.extract(
        r"^([0-9]+(?:\.[0-9]+)?)",
        expand=False,
    ),
    errors="coerce",
)

In [65]:
# 유효한 연비값으로 사용하지 않기로 한 0값을 결측값으로 변경합니다.
df_preprocessed.loc[
    df_preprocessed["mileage"].eq(0),
    "mileage",
] = pd.NA

In [66]:
# mileage가 숫자형으로 변환되었는지 확인합니다.
df_preprocessed["mileage"].dtype

dtype('float64')

In [67]:
# 전처리 후 mileage 결측치 수를 확인합니다.
df_preprocessed["mileage"].isna().sum()

np.int64(223)

In [68]:
# 전처리 후 mileage 0값이 남아 있는지 확인합니다.
df_preprocessed["mileage"].eq(0).sum()

np.int64(0)

In [69]:
# 0값을 결측 처리한 후 mileage 숫자값의 분포를 확인합니다.
df_preprocessed["mileage"].describe()

count    6703.000000
mean       19.508869
std         3.947453
min         9.000000
25%        16.800000
50%        19.490000
75%        22.540000
max        42.000000
Name: mileage, dtype: float64

In [70]:
# mileage 처리 전후 상태를 한 번에 비교합니다.
pd.Series({
    "처리 전 mileage 결측": df_clean["mileage"].isna().sum(),
    "처리 후 mileage 결측": df_preprocessed["mileage"].isna().sum(),
    "처리 후 mileage 0값": df_preprocessed["mileage"].eq(0).sum(),
})

처리 전 mileage 결측    208
처리 후 mileage 결측    223
처리 후 mileage 0값      0
dtype: int64

In [71]:
# mileage를 제외한 다른 컬럼이 변경되지 않았는지 확인합니다.
other_columns = df_clean.columns.difference(["mileage"])

df_clean[other_columns].equals(
    df_preprocessed[other_columns]
)

True

In [72]:
# mileage 전처리 후에도 전체 행과 열 수가 유지되는지 확인합니다.
df_preprocessed.shape

(6926, 13)

In [73]:
# mileage 전처리로 완전 중복 행이 생겼는지 확인합니다.
df_preprocessed.duplicated().sum()

np.int64(0)

In [74]:
# engine의 숫자 부분을 확인용 숫자 Series로 변환합니다.
engine_number = pd.to_numeric(
    df_preprocessed["engine"].str.extract(
        r"^([0-9]+(?:\.[0-9]+)?)",
        expand=False,
    ),
    errors="coerce",
)

In [75]:
# engine 숫자값의 기본 분포를 확인합니다.
engine_number.describe()

count    6718.000000
mean     1430.891337
std       493.493277
min       624.000000
25%      1197.000000
50%      1248.000000
75%      1498.000000
max      3604.000000
Name: engine, dtype: float64

In [76]:
# 원래 engine이 결측이 아닌 행 중 숫자 변환에 실패한 행을 확인합니다.
engine_non_null_mask = df_preprocessed["engine"].notna()

engine_conversion_failure_mask = (
    engine_non_null_mask
    & engine_number.isna()
)

engine_conversion_failure_mask.sum()

np.int64(0)

In [77]:
# engine 숫자 변환에 실패한 원본 행을 확인합니다.
df_preprocessed.loc[
    engine_conversion_failure_mask,
    ["name", "year", "engine"],
]

,name,year,engine


In [78]:
# engine 숫자값이 0 이하인 행 수를 확인합니다.
engine_number.le(0).sum()

np.int64(0)

In [79]:
# engine 숫자값이 0 이하인 원본 행을 확인합니다.
df_preprocessed.loc[
    engine_number.le(0),
    ["name", "year", "fuel", "engine"],
]

,name,year,fuel,engine


In [80]:
# 양수인 engine 고유값 중 가장 작은 10개를 확인합니다.
(
    engine_number[engine_number.gt(0)]
    .drop_duplicates()
    .sort_values()
    .head(10)
)

363     624.0
492     793.0
7       796.0
221     799.0
406     814.0
1327    909.0
503     936.0
11      993.0
986     995.0
36      998.0
Name: engine, dtype: float64

In [81]:
# 가장 작은 engine 값 5종에 해당하는 차량을 값별 최대 5행씩 확인합니다.
smallest_engine_values = (
    engine_number[engine_number.gt(0)]
    .drop_duplicates()
    .sort_values()
    .head(5)
)

smallest_engine_mask = engine_number.isin(smallest_engine_values)
smallest_engine_example_index = (
    engine_number[smallest_engine_mask]
    .groupby(engine_number[smallest_engine_mask])
    .head(5)
    .sort_values()
    .index
)

df_preprocessed.loc[
    smallest_engine_example_index,
    ["name", "year", "fuel", "engine"],
]

,name,year,fuel,engine
363,Tata Nano STD,2012,Petrol,624 CC
1267,Tata Nano Cx BSIV,2010,Petrol,624 CC
1217,Tata Nano CX,2013,Petrol,624 CC
709,Tata Nano Cx,2011,Petrol,624 CC
1425,Tata Nano XTA,2015,Petrol,624 CC
492,Maruti Celerio LDi,2016,Diesel,793 CC
1333,Maruti Celerio VDi,2015,Diesel,793 CC
2035,Maruti Celerio ZDi,2015,Diesel,793 CC
7372,Maruti Celerio VDi,2016,Diesel,793 CC
5016,Maruti Celerio ZDi,2015,Diesel,793 CC


In [82]:
# engine의 기존 결측치 수를 다시 확인합니다.
df_preprocessed["engine"].isna().sum()

np.int64(208)

In [83]:
# engine 조사 후에도 전체 행과 열 수가 유지되는지 확인합니다.
df_preprocessed.shape

(6926, 13)

In [84]:
# engine 조사 후에도 완전 중복 행이 없는지 확인합니다.
df_preprocessed.duplicated().sum()

np.int64(0)

In [85]:
# engine에서 숫자 부분만 추출해 숫자형으로 변환합니다.
df_preprocessed["engine"] = pd.to_numeric(
    df_preprocessed["engine"].str.extract(
        r"^([0-9]+(?:\.[0-9]+)?)",
        expand=False,
    ),
    errors="coerce",
)

In [86]:
# engine이 숫자형으로 변환되었는지 확인합니다.
df_preprocessed["engine"].dtype

dtype('float64')

In [87]:
# 숫자 변환 후 engine 결측치 수를 확인합니다.
df_preprocessed["engine"].isna().sum()

np.int64(208)

In [88]:
# 숫자 변환 후 engine 값의 분포를 확인합니다.
df_preprocessed["engine"].describe()

count    6718.000000
mean     1430.891337
std       493.493277
min       624.000000
25%      1197.000000
50%      1248.000000
75%      1498.000000
max      3604.000000
Name: engine, dtype: float64

In [89]:
# 숫자 변환 후 engine에 0 이하 값이 있는지 확인합니다.
df_preprocessed["engine"].le(0).sum()

np.int64(0)

In [90]:
# 문자열 단위가 제거된 engine 값의 예시를 확인합니다.
df_preprocessed.loc[
    df_preprocessed["engine"].notna(),
    ["name", "engine"],
].head(10)

,name,engine
0,Maruti Swift Dzire VDI,1248.0
1,Skoda Rapid 1.5 TDI Ambition,1498.0
2,Honda City 2017-2020 EXi,1497.0
3,Hyundai i20 Sportz Diesel,1396.0
4,Maruti Swift VXI BSIII,1298.0
5,Hyundai Xcent 1.2 VTVT E Plus,1197.0
6,Maruti Wagon R LXI DUO BSIII,1061.0
7,Maruti 800 DX BSII,796.0
8,Toyota Etios VXD,1364.0
9,Ford Figo Diesel Celebration Edition,1399.0


In [91]:
# engine 전처리 후에도 mileage 상태가 유지되는지 확인합니다.
pd.Series({
    "mileage 결측": df_preprocessed["mileage"].isna().sum(),
    "mileage 0값": df_preprocessed["mileage"].eq(0).sum(),
})

mileage 결측    223
mileage 0값      0
dtype: int64

In [92]:
# engine 전처리 후에도 전체 행과 열 수가 유지되는지 확인합니다.
df_preprocessed.shape

(6926, 13)

In [93]:
# engine 전처리 후 완전 중복 행 수를 확인합니다.
df_preprocessed.duplicated().sum()

np.int64(0)